In [1]:
# ==========================================================
# VALUE-AT-RISK (VaR) & RISK ANALYTICS DASHBOARD
# Author: Zander Felder
# ==========================================================

!pip -q install yfinance plotly scipy

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import yfinance as yf

import plotly.express as px
import plotly.graph_objects as go

import matplotlib.pyplot as plt

from scipy.stats import norm

from pathlib import Path

START_DATE = "2021-01-01"

END_DATE = None

CONFIDENCE_LEVEL = 0.95

MONTE_CARLO_SIMULATIONS = 100000

TICKERS = [

    "AAPL",
    "MSFT",
    "NVDA",
    "AMZN",
    "META",
    "GOOGL",
    "AVGO",
    "JPM",
    "LLY",
    "XOM"

]

print("="*60)
print("Value-at-Risk Dashboard Initialized")
print("="*60)

print(f"Assets: {len(TICKERS)}")
print(f"Confidence Level: {CONFIDENCE_LEVEL:.0%}")
print(f"Monte Carlo Simulations: {MONTE_CARLO_SIMULATIONS:,}")

Value-at-Risk Dashboard Initialized
Assets: 10
Confidence Level: 95%
Monte Carlo Simulations: 100,000


In [2]:
print("="*60)
print("Downloading Market Data...")
print("="*60)

prices = yf.download(

    TICKERS,

    start=START_DATE,

    end=END_DATE,

    auto_adjust=True,

    progress=False

)["Close"]

prices = prices.dropna(axis=1, thresh=len(prices)*0.95)

prices = prices.ffill().bfill()

prices = prices.dropna()

TICKERS = prices.columns.tolist()

returns = prices.pct_change().dropna()

weights = np.ones(len(TICKERS))

weights /= weights.sum()

portfolio_returns = returns @ weights

print()

print(f"Assets Loaded: {len(TICKERS)}")

print(f"Trading Days: {len(portfolio_returns)}")

display(prices.tail())

fig = px.line(

    prices,

    title="Historical Asset Prices"

)

fig.update_layout(

    template="plotly_white",

    height=600

)

fig.show()


Assets Loaded: 10
Trading Days: 1380


Ticker,AAPL,AMZN,AVGO,GOOGL,JPM,LLY,META,MSFT,NVDA,XOM
Date,,,,,,,,,,
2026-06-29,281.739990,240.139999,372.450012,353.649994,329.390015,1229.930054,562.599976,368.570007,194.970001,136.059998
2026-06-30,289.359985,238.339996,377.750000,357.369995,327.329987,1199.430054,563.289978,373.019989,200.089996,136.720001
2026-07-01,294.380005,241.699997,369.339996,361.209991,334.070007,1191.739990,612.909973,384.279999,197.580002,136.279999
2026-07-02,308.630005,242.669998,360.450012,359.910004,334.470001,1213.910034,582.900024,390.489990,194.830002,137.089996
2026-07-06,312.690002,244.741501,375.605011,363.980011,336.304993,1205.979980,592.184998,385.179993,196.419998,136.485001


In [3]:
print("="*60)
print("Engineering Risk Features...")
print("="*60)

daily_mean = portfolio_returns.mean()

daily_std = portfolio_returns.std()

annual_return = daily_mean * 252

annual_volatility = daily_std * np.sqrt(252)

cumulative_returns = (1 + portfolio_returns).cumprod()

running_max = cumulative_returns.cummax()

drawdown = (
    cumulative_returns -
    running_max
) / running_max

rolling_volatility = (
    portfolio_returns
    .rolling(63)
    .std()
    * np.sqrt(252)
)

rolling_return = (
    portfolio_returns
    .rolling(63)
    .mean()
    * 252
)

rolling_sharpe = (
    rolling_return
) / rolling_volatility

risk_summary = pd.DataFrame({

    "Metric":[

        "Annual Return",
        "Annual Volatility",
        "Average Daily Return",
        "Daily Volatility",
        "Worst Daily Return",
        "Best Daily Return"

    ],

    "Value":[

        annual_return,
        annual_volatility,
        daily_mean,
        daily_std,
        portfolio_returns.min(),
        portfolio_returns.max()

    ]

})

display(risk_summary)

fig = px.line(

    x=rolling_volatility.index,

    y=rolling_volatility,

    title="Rolling Annualized Volatility"

)

fig.update_layout(

    template="plotly_white",

    height=550

)

fig.show()

Engineering Risk Features...


,Metric,Value
0,Annual Return,0.313271
1,Annual Volatility,0.223935
2,Average Daily Return,0.001243
3,Daily Volatility,0.014107
4,Worst Daily Return,-0.067670
5,Best Daily Return,0.116093


In [4]:
print("="*60)
print("Historical Value-at-Risk")
print("="*60)

historical_var = np.percentile(

    portfolio_returns,

    (1-CONFIDENCE_LEVEL)*100

)

historical_cvar = portfolio_returns[

    portfolio_returns <= historical_var

].mean()

print()

print(f"{CONFIDENCE_LEVEL:.0%} Historical VaR : {historical_var:.2%}")

print(f"{CONFIDENCE_LEVEL:.0%} Historical CVaR: {historical_cvar:.2%}")

fig = px.histogram(

    portfolio_returns,

    nbins=80,

    title="Historical Portfolio Return Distribution"

)

fig.add_vline(

    x=historical_var,

    line_color="red",

    line_width=3,

    annotation_text=f"VaR {historical_var:.2%}"

)

fig.add_vline(

    x=historical_cvar,

    line_color="orange",

    line_width=3,

    annotation_text=f"CVaR {historical_cvar:.2%}"

)

fig.update_layout(

    template="plotly_white",

    height=650

)

fig.show()

historical_results = pd.DataFrame({

    "Confidence":[CONFIDENCE_LEVEL],

    "Historical VaR":[historical_var],

    "Historical CVaR":[historical_cvar]

})

display(historical_results)

Historical Value-at-Risk

95% Historical VaR : -2.25%
95% Historical CVaR: -3.18%


,Confidence,Historical VaR,Historical CVaR
0,0.95,-0.022486,-0.031771


In [5]:
print("="*60)
print("Parametric Value-at-Risk")
print("="*60)

z_score = norm.ppf(1 - CONFIDENCE_LEVEL)

parametric_var = (
    daily_mean +
    z_score * daily_std
)

parametric_cvar = (
    daily_mean -
    daily_std *
    (
        norm.pdf(z_score)
        /
        (1 - CONFIDENCE_LEVEL)
    )
)

print()

print(f"{CONFIDENCE_LEVEL:.0%} Parametric VaR : {parametric_var:.2%}")
print(f"{CONFIDENCE_LEVEL:.0%} Parametric CVaR: {parametric_cvar:.2%}")

x = np.linspace(

    portfolio_returns.min()*1.5,

    portfolio_returns.max()*1.5,

    1000

)

pdf = norm.pdf(

    x,

    daily_mean,

    daily_std

)

fig = go.Figure()

fig.add_trace(

    go.Scatter(

        x=x,

        y=pdf,

        mode="lines",

        name="Normal Distribution"

    )

)

fig.add_vline(

    x=parametric_var,

    line_color="red",

    line_width=3,

    annotation_text="VaR"

)

fig.add_vline(

    x=parametric_cvar,

    line_color="orange",

    line_width=3,

    annotation_text="CVaR"

)

fig.update_layout(

    title="Parametric Value-at-Risk",

    template="plotly_white",

    height=650

)

fig.show()

parametric_results = pd.DataFrame({

    "Confidence":[CONFIDENCE_LEVEL],

    "Parametric VaR":[parametric_var],

    "Parametric CVaR":[parametric_cvar]

})

display(parametric_results)

Parametric Value-at-Risk

95% Parametric VaR : -2.20%
95% Parametric CVaR: -2.79%


,Confidence,Parametric VaR,Parametric CVaR
0,0.95,-0.02196,-0.027855


In [6]:
print("="*60)
print("Monte Carlo Value-at-Risk")
print("="*60)

np.random.seed(42)

simulated_returns = np.random.normal(

    daily_mean,

    daily_std,

    MONTE_CARLO_SIMULATIONS

)

monte_carlo_var = np.percentile(

    simulated_returns,

    (1-CONFIDENCE_LEVEL)*100

)

monte_carlo_cvar = simulated_returns[

    simulated_returns <= monte_carlo_var

].mean()

print()

print(f"{CONFIDENCE_LEVEL:.0%} Monte Carlo VaR : {monte_carlo_var:.2%}")
print(f"{CONFIDENCE_LEVEL:.0%} Monte Carlo CVaR: {monte_carlo_cvar:.2%}")

fig = px.histogram(

    simulated_returns,

    nbins=120,

    title="Monte Carlo Return Distribution"

)

fig.add_vline(

    x=monte_carlo_var,

    line_color="red",

    line_width=3,

    annotation_text="VaR"

)

fig.add_vline(

    x=monte_carlo_cvar,

    line_color="orange",

    line_width=3,

    annotation_text="CVaR"

)

fig.update_layout(

    template="plotly_white",

    height=650

)

fig.show()

monte_carlo_results = pd.DataFrame({

    "Confidence":[CONFIDENCE_LEVEL],

    "Monte Carlo VaR":[monte_carlo_var],

    "Monte Carlo CVaR":[monte_carlo_cvar]

})

display(monte_carlo_results)

comparison = pd.DataFrame({

    "Method":[

        "Historical",

        "Parametric",

        "Monte Carlo"

    ],

    "VaR":[

        historical_var,

        parametric_var,

        monte_carlo_var

    ],

    "CVaR":[

        historical_cvar,

        parametric_cvar,

        monte_carlo_cvar

    ]

})

print()

print("Risk Model Comparison")

display(comparison)

Monte Carlo Value-at-Risk

95% Monte Carlo VaR : -2.19%
95% Monte Carlo CVaR: -2.79%


,Confidence,Monte Carlo VaR,Monte Carlo CVaR
0,0.95,-0.021941,-0.027884



Risk Model Comparison


,Method,VaR,CVaR
0,Historical,-0.022486,-0.031771
1,Parametric,-0.021960,-0.027855
2,Monte Carlo,-0.021941,-0.027884


In [7]:
print("="*60)
print("Risk Analytics Dashboard")
print("="*60)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=cumulative_returns.index,
        y=cumulative_returns,
        name="Portfolio Value",
        line=dict(width=2)
    )
)

fig.update_layout(
    title="Portfolio Growth",
    template="plotly_white",
    height=600,
    xaxis_title="Date",
    yaxis_title="Growth"
)

fig.show()

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=drawdown.index,
        y=drawdown,
        fill="tozeroy",
        name="Drawdown"
    )
)

fig.update_layout(
    title="Portfolio Drawdown",
    template="plotly_white",
    height=500,
    xaxis_title="Date",
    yaxis_title="Drawdown"
)

fig.show()

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=rolling_volatility.index,
        y=rolling_volatility,
        name="Rolling Volatility"
    )
)

fig.update_layout(
    title="63-Day Rolling Volatility",
    template="plotly_white",
    height=500
)

fig.show()

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=rolling_sharpe.index,
        y=rolling_sharpe,
        name="Rolling Sharpe"
    )
)

fig.update_layout(
    title="63-Day Rolling Sharpe Ratio",
    template="plotly_white",
    height=500
)

fig.show()

fig = px.bar(

    comparison,

    x="Method",

    y="VaR",

    color="Method",

    title="Value-at-Risk Comparison"

)

fig.update_layout(

    template="plotly_white",

    height=550

)

fig.show()

fig = px.bar(

    comparison,

    x="Method",

    y="CVaR",

    color="Method",

    title="Conditional Value-at-Risk Comparison"

)

fig.update_layout(

    template="plotly_white",

    height=550

)

fig.show()

Risk Analytics Dashboard


In [10]:
Path("outputs").mkdir(exist_ok=True)

comparison.to_csv(

    "outputs/var_comparison.csv",

    index=False

)

historical_results.to_csv(

    "outputs/historical_var.csv",

    index=False

)

parametric_results.to_csv(

    "outputs/parametric_var.csv",

    index=False

)

monte_carlo_results.to_csv(

    "outputs/monte_carlo_var.csv",

    index=False

)

risk_summary.to_csv(

    "outputs/risk_statistics.csv",

    index=False

)

print("="*60)
print("Export Complete")
print("="*60)

for file in sorted(Path("outputs").glob("*")):

    print(file.name)

display(comparison)

display(risk_summary)

Export Complete
historical_var.csv
monte_carlo_var.csv
parametric_var.csv
risk_statistics.csv
var_comparison.csv


,Method,VaR,CVaR
0,Historical,-0.022486,-0.031771
1,Parametric,-0.021960,-0.027855
2,Monte Carlo,-0.021941,-0.027884


,Metric,Value
0,Annual Return,0.313271
1,Annual Volatility,0.223935
2,Average Daily Return,0.001243
3,Daily Volatility,0.014107
4,Worst Daily Return,-0.067670
5,Best Daily Return,0.116093


In [9]:
print("="*70)
print("             VALUE-AT-RISK RISK ANALYTICS SUMMARY")
print("="*70)

print(f"\nAssets Analyzed: {len(TICKERS)}")
print(f"Trading Days: {len(portfolio_returns):,}")
print(f"Confidence Level: {CONFIDENCE_LEVEL:.0%}")
print(f"Monte Carlo Simulations: {MONTE_CARLO_SIMULATIONS:,}")

print("\nHistorical Risk")
print(f"VaR  : {historical_var:.2%}")
print(f"CVaR : {historical_cvar:.2%}")

print("\nParametric Risk")
print(f"VaR  : {parametric_var:.2%}")
print(f"CVaR : {parametric_cvar:.2%}")

print("\nMonte Carlo Risk")
print(f"VaR  : {monte_carlo_var:.2%}")
print(f"CVaR : {monte_carlo_cvar:.2%}")

print("\nAnnual Return :", f"{annual_return:.2%}")
print("Annual Volatility :", f"{annual_volatility:.2%}")

display(comparison)

display(risk_summary)

print("="*70)
print("Risk Analysis Complete")
print("="*70)

             VALUE-AT-RISK RISK ANALYTICS SUMMARY

Assets Analyzed: 10
Trading Days: 1,380
Confidence Level: 95%
Monte Carlo Simulations: 100,000

Historical Risk
VaR  : -2.25%
CVaR : -3.18%

Parametric Risk
VaR  : -2.20%
CVaR : -2.79%

Monte Carlo Risk
VaR  : -2.19%
CVaR : -2.79%

Annual Return : 31.33%
Annual Volatility : 22.39%


,Method,VaR,CVaR
0,Historical,-0.022486,-0.031771
1,Parametric,-0.021960,-0.027855
2,Monte Carlo,-0.021941,-0.027884


,Metric,Value
0,Annual Return,0.313271
1,Annual Volatility,0.223935
2,Average Daily Return,0.001243
3,Daily Volatility,0.014107
4,Worst Daily Return,-0.067670
5,Best Daily Return,0.116093


Risk Analysis Complete


In [8]:
Path("screenshots").mkdir(exist_ok=True)

plt.figure(figsize=(12,6))
plt.plot(cumulative_returns)
plt.title("Portfolio Growth")
plt.grid(True)
plt.tight_layout()
plt.savefig("screenshots/portfolio_growth.png",dpi=300)
plt.close()

plt.figure(figsize=(12,5))
plt.plot(drawdown)
plt.title("Portfolio Drawdown")
plt.grid(True)
plt.tight_layout()
plt.savefig("screenshots/portfolio_drawdown.png",dpi=300)
plt.close()

plt.figure(figsize=(8,6))
plt.hist(portfolio_returns,bins=60)
plt.axvline(historical_var,color="red")
plt.title("Historical VaR")
plt.tight_layout()
plt.savefig("screenshots/historical_var.png",dpi=300)
plt.close()

plt.figure(figsize=(8,6))
plt.hist(simulated_returns,bins=80)
plt.axvline(monte_carlo_var,color="red")
plt.title("Monte Carlo VaR")
plt.tight_layout()
plt.savefig("screenshots/monte_carlo_var.png",dpi=300)
plt.close()

plt.figure(figsize=(8,5))
plt.bar(comparison["Method"],comparison["VaR"])
plt.title("VaR Comparison")
plt.tight_layout()
plt.savefig("screenshots/var_comparison.png",dpi=300)
plt.close()

print("="*70)
print("PROJECT COMPLETE")
print("="*70)

print("\nOutputs")
for f in sorted(Path("outputs").glob("*")):
    print("•",f.name)

print("\nScreenshots")
for f in sorted(Path("screenshots").glob("*")):
    print("•",f.name)

print("\nNotebook Status: COMPLETE")

PROJECT COMPLETE

Outputs

Screenshots
• historical_var.png
• monte_carlo_var.png
• portfolio_drawdown.png
• portfolio_growth.png
• var_comparison.png

Notebook Status: COMPLETE
